In [4]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
print('use device : ', device)

use device :  cpu


In [ ]:
import torch
import time

size = 10000

a_cpu = torch.rand(size, size)
b_cpu = torch.rand(size, size)

start_time = time.time()
_ = torch.mm(a_cpu, b_cpu)
cpu_time = time.time() - start_time
# matmul = 행렬곱
# mat / mul
print('cpu matmul time : ', cpu_time)

if torch.cuda.is_available():
    a_gpu = a_cpu.to(device)
    b_gpu = b_cpu.to(device)

    # synchronize : 동기화(시작 시간을 맞춤)
    # gpu 성능 측정을 위해 동기화
    # cpu는 자동으로 동기화해서 한줄씩 실행
    # 동기 실행 : 작업이 끝나면 넘어가기
    # 비동기 실행(백그라운드) : 작업을 다른 친구(gpu)에게 시키고 다음으로 넘어감(결과는 필요할떄 확인)
    torch.cuda.synchronize()
    start_time = time.time()
    _ = torch.mm(a_gpu, b_gpu)
    torch.cuda.synchronize()
    gpu_time = time.time() - start_time
    print('gpu matmul time : ', gpu_time)

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset

X_train = torch.randn(100, 10)
y_train = torch.randn(100, 1)
dataset = TensorDataset(X_train, y_train)
dataloader = DataLoader(dataset, batch_size = 32, shuffle = True)

class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = SimpleNN(input_size=10, hidden_size =20, output_size=1)
print(model)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(18):
    for batch in dataloader:
        inputs, targets = batch
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
    print(f'Epoch : {epoch+1}, Loss : {loss.item()}')

torch.save(model.state_dict(), 'model.pth')
model.load_state_dict(torch.load('model.pth'))

if torch.isnan(loss).any():
    print("Loss에 null 값이 포함되어 있다, learning rate를 줄여라")
    for param_group in optimizer.param_groups:
        param_group['lr'] *= 0.1

# 데이터 증강 기법 적용(이미지 데이터의 경우)
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

SimpleNN(
  (fc1): Linear(in_features=10, out_features=20, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=20, out_features=1, bias=True)
)
Epoch : 1, Loss : 0.6900784373283386
Epoch : 2, Loss : 1.6391758918762207
Epoch : 3, Loss : 0.5450261831283569
Epoch : 4, Loss : 0.9715118408203125
Epoch : 5, Loss : 1.7967309951782227
Epoch : 6, Loss : 0.6967663168907166
Epoch : 7, Loss : 1.1672602891921997
Epoch : 8, Loss : 2.3424360752105713
Epoch : 9, Loss : 0.8981301784515381
Epoch : 10, Loss : 1.4394495487213135
Epoch : 11, Loss : 1.4474272727966309
Epoch : 12, Loss : 0.20138317346572876
Epoch : 13, Loss : 0.03568015992641449
Epoch : 14, Loss : 0.1389748603105545
Epoch : 15, Loss : 1.028024673461914
Epoch : 16, Loss : 0.7310059666633606
Epoch : 17, Loss : 1.0361087322235107
Epoch : 18, Loss : 0.11173668503761292


In [5]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform = transform, download = True)

test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size =64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size =1000, shuffle=False)

class MLP_Final(nn.Module):
    def __init__(self):
        super(MLP_Final, self).__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = MLP_Final().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("최종 모델 학습 중...")

for epoch in range(3):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 300 == 0:
            print(f'Epoch : {epoch+1}, Batch : {batch_idx}, Loss : {loss.item():.4f}')

최종 모델 학습 중...
Epoch : 1, Batch : 0, Loss : 2.3081
Epoch : 1, Batch : 300, Loss : 0.2727
Epoch : 1, Batch : 600, Loss : 0.1467
Epoch : 1, Batch : 900, Loss : 0.0909
Epoch : 2, Batch : 0, Loss : 0.0570
Epoch : 2, Batch : 300, Loss : 0.1655
Epoch : 2, Batch : 600, Loss : 0.0353
Epoch : 2, Batch : 900, Loss : 0.0920
Epoch : 3, Batch : 0, Loss : 0.0226
Epoch : 3, Batch : 300, Loss : 0.1048
Epoch : 3, Batch : 600, Loss : 0.0495
Epoch : 3, Batch : 900, Loss : 0.0807
